# Exploring the Dataset: Building the HTTP Events Table

**Goal:** Understand how we go from a raw Apache error log annotation file to a relational database table.

This notebook walks through:
1. Loading `apache` (the raw JSON Lines file containing annotated HTTP event records)
2. Examining what a single record looks like
3. Transforming the data into a Pandas DataFrame
4. Mapping it to our planned `http_events` database table schema

---

**Dataset:** AIT Log Data Set V2.0 - russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.  

**Default:** Uses the absolute path to the dataset on this machine:
```
C:/Users/ishaanshetty/DATA-201/russellmitchell/
```

If you're running this on a different machine, update `DATASET_ROOT` to point to your local copy of the `russellmitchell/` folder.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path(r"C:/Users/ishaanshetty/DATA-201/russellmitchell")

# The apache error log lives here within the dataset:
# logs/intranet_server/apache2/intranet.smith.russellmitchell.com-error_log.2

# Verify the path exists
if not DATASET_ROOT.exists():
    print(f"ERROR: Dataset not found at: {DATASET_ROOT.resolve()}")
    print("")
    print("Expected directory structure:")
    print("  <workspace>/data-201-security-log-analysis/notebooks/  <-- you are here")
    print("  <workspace>/russellmitchell/                          <-- dataset should be here")
    print("")
    print("Fix: Update DATASET_ROOT above to point to your russellmitchell/ folder.")
else:
    print(f"Dataset found at: {DATASET_ROOT.resolve()}")

Dataset found at: C:\Users\ishaanshetty\DATA-201\russellmitchell


## 1. Load the Raw Apache File

The file `apache` is derived from `logs/intranet_server/apache2/intranet.smith.russellmitchell.com-error_log.2` in the russellmitchell dataset.  
It contains annotated Apache error log records in JSON Lines format - each line is one HTTP event record.  
This is the source for our **`http_events`** database table.

In [1]:
import re

import pandas as pd

annotation_path = (
    DATASET_ROOT
    / "labels"
    / "intranet_server"
    / "logs"
    / "apache2"
    / "intranet.smith.russellmitchell.com-error_log.2"
)
raw_log_path = (
    DATASET_ROOT
    / "gather"
    / "intranet_server"
    / "logs"
    / "apache2"
    / "intranet.smith.russellmitchell.com-error_log.2"
)


def parse_error_log(annotation_path, raw_log_path):
    # Load annotation (JSON Lines)
    df_ann = pd.read_json(annotation_path, lines=True)
    print(f"Loaded {len(df_ann)} annotation records")
    print(f"\nColumns: {list(df_ann.columns)}")
    print("\nUnique label categories found:")
    all_labels = sorted({lbl for labels in df_ann["labels"] for lbl in labels})
    for i, label in enumerate(all_labels, 1):
        print(f"  {i:2d}. {label}")

    # Parse raw error log: [timestamp] [module:level] [pid N] [client IP:PORT] message
    error_pattern = re.compile(
        r"^\[([^\]]+)\]\s+\[([^\]]+)\](?:\s+\[pid \d+\])?(?:\s+\[client ([^\]]+)\])?\s+(.*)$"
    )
    raw_rows = []
    with open(raw_log_path) as f:
        for lineno, line in enumerate(f, start=1):
            m = error_pattern.match(line.strip())
            if m:
                ts, level, client, msg = m.groups()
                raw_rows.append(
                    {
                        "line": lineno,
                        "timestamp_raw": ts,
                        "log_level": level,
                        "client_ip": client.split(":")[0] if client else None,
                        "message": msg,
                    }
                )

    df_raw = pd.DataFrame(raw_rows)
    df_raw["event_timestamp"] = pd.to_datetime(
        df_raw["timestamp_raw"], format="%a %b %d %H:%M:%S.%f %Y", errors="coerce"
    )

    # Join annotation with parsed log fields on line number
    df_merged = df_raw.merge(df_ann[["line", "labels", "rules"]], on="line", how="inner")
    print(f"\nJoined {len(df_merged)} annotated log records with parsed fields")
    return df_merged


df_merged = parse_error_log(annotation_path, raw_log_path)

Loaded 35 annotation records

Columns: ['line', 'labels', 'rules']

Unique label categories found:
   1. attacker_http
   2. dirb
   3. foothold
   4. wpscan

Joined 35 annotated log records with parsed fields


## 2. Examine a Single Record

Let's look at what data exists for one record. We'll pick the first entry to see the full raw structure before we flatten anything.

In [2]:
import json

example_record = df_merged.iloc[0].to_dict()

print("Raw data for record 0:\n")
print(json.dumps(example_record, indent=2))

Raw data for record 0:

{
  "line": 2,
  "labels": [
    "attacker_http",
    "foothold",
    "dirb"
  ],
  "rules": {
    "attacker_http": [
      "attacker.foothold.apache.error",
      "attacker.foothold.apache.access_error"
    ],
    "foothold": [
      "attacker.foothold.apache.error",
      "attacker.foothold.apache.access_error"
    ],
    "dirb": [
      "attacker.dirb.time"
    ]
  }
}


### What do these fields mean?

| Field | What It Is | Example |
|-------|-----------|--------|
| `line` | The original line number from the Apache error log file | `1` |
| `labels` | List of attack-phase tags that apply to this log line | `[attacker_http, foothold, dirb]` |
| `rules` | Dictionary mapping each label to the specific detection rules that matched | `{"dirb": ["attacker.foothold.apache.error.dirb"]}` |

The `labels` field tells us **what kind of attack activity** this log line represents.  
The `rules` field tells us **exactly which detection signature** fired for each label.  
Not all records will have all four label categories - a record tagged only with `wpscan` will only have `wpscan` in its `rules` dict.

### 2.1 Log format structure

The Apache error log uses this line format:
```
[timestamp] [module:level] [pid N] [client IP:PORT] message
```
The regex in the load cell extracts four groups: timestamp, log level (module:severity), client IP (stripping the port), and the message body.


### 2.2 Parsing strategy

| Field | Source | Notes |
|-------|--------|-------|
| `timestamp_raw` | Group 1 - full timestamp string | e.g. `Mon Jan 24 03:57:26.698653 2022` |
| `log_level` | Group 2 - `module:severity` | e.g. `authz_core:error`, `php7:error` |
| `client_ip` | Group 3 - IP:PORT, port stripped | `rsplit(':', 1)[0]` handles both IPv4 and IPv6 |
| `message` | Group 4 - remainder of line | Free text, may contain Apache error codes (AH#####) or PHP errors |
| `event_timestamp` | Parsed from `timestamp_raw` | `pd.to_datetime` with `errors='coerce'` |


## 3. Field-by-Field Exploration

### 3.1 log_level (module and severity)


In [ ]:
print("=== log_level distribution ===")
for level, count in df_merged["log_level"].value_counts().items():
    module, _, severity = level.partition(":")
    print(f"  {level:30s}  {count:3d}   module={module}  severity={severity}")

### 3.2 client_ip


In [ ]:
print("=== client_ip distribution ===")
for ip, count in df_merged["client_ip"].value_counts().items():
    print(f"  {ip}: {count}")
print()
print(f"Null client_ip: {df_merged['client_ip'].isna().sum()}")

### 3.3 event_timestamp


In [ ]:
print("=== Timestamp range ===")
print(f"  Earliest: {df_merged['event_timestamp'].min()}")
print(f"  Latest:   {df_merged['event_timestamp'].max()}")
print(f"  Span:     {df_merged['event_timestamp'].max() - df_merged['event_timestamp'].min()}")
print(f"  Null timestamps: {df_merged['event_timestamp'].isna().sum()}")

### 3.4 message


In [ ]:
print("=== Message prefix distribution (first token) ===")

df_merged["_msg_prefix"] = df_merged["message"].str.extract(
    r"^(AH\d+|PHP\s+\w+|script)", expand=False
)
for prefix, count in df_merged["_msg_prefix"].value_counts().items():
    print(f"  {prefix}: {count}")
print(f"  (no recognised prefix): {df_merged['_msg_prefix'].isna().sum()}")
df_merged.drop(columns=["_msg_prefix"], inplace=True)

### 3.5 labels and rules


In [ ]:
from collections import Counter

print("=== Label co-occurrence ===")
label_combos = Counter(tuple(sorted(x)) for x in df_merged["labels"])
for combo, count in label_combos.most_common():
    print(f"  {list(combo)}: {count}")

print()
print("=== Labels per record ===")
labels_per_row = Counter(len(x) for x in df_merged["labels"])
for n, count in sorted(labels_per_row.items()):
    print(f"  {n} label(s): {count} records")

print()
print("=== Rule keys per record ===")
rule_keys_per_row = Counter(len(x) if isinstance(x, dict) else 0 for x in df_merged["rules"])
for n, count in sorted(rule_keys_per_row.items()):
    print(f"  {n} rule key(s): {count} records")

## 4. Label Integration

Load the ground truth labels for this log. All 35 records are labeled - cross-reference them with the parsed fields.


In [ ]:
import json
from collections import Counter

labels_list = []
with open(annotation_path) as f:
    for line in f:
        labels_list.append(json.loads(line))

print(f"Labeled lines: {len(labels_list)}")
print(f"Labeled line numbers: {[lbl['line'] for lbl in labels_list]}")
print()

# Label distribution
label_counter = Counter()
for lbl in labels_list:
    for label_name in lbl["labels"]:
        label_counter[label_name] += 1

print("=== Label distribution ===")
for label_name, count in label_counter.most_common():
    print(f"  {label_name}: {count} lines")

print()
print("=== Labels per line ===")
labels_per_line = Counter(len(lbl["labels"]) for lbl in labels_list)
for n, count in sorted(labels_per_line.items()):
    print(f"  {n} labels: {count} lines")

print()
print("=== Rule distribution ===")
rule_counter = Counter()
for lbl in labels_list:
    for rule_list in lbl["rules"].values():
        for rule_name in rule_list:
            rule_counter[rule_name] += 1
for rule_name, count in rule_counter.most_common():
    print(f"  {rule_name}: {count}")

## 5. Raw 1:1 DataFrame

Build the definitive raw DataFrame with one row per labeled log line, preserving all parsed fields as they will be stored in the `http_events` table.


In [ ]:
# Rename columns to match the http_events schema
df_raw = df_merged[
    ["line", "event_timestamp", "log_level", "client_ip", "message", "labels", "rules"]
].copy()
df_raw.columns = [
    "event_id",
    "event_timestamp",
    "log_level",
    "client_ip",
    "message",
    "http_event_category",
    "http_signature_matches",
]
df_raw.insert(0, "http_event_id", range(1, len(df_raw) + 1))

print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print()
print("=== First 5 rows ===")
print(df_raw.head().to_string())
print()
print("=== Last 5 rows ===")
print(df_raw.tail().to_string())

## 6. Understand the Label Distribution

The `labels` field categorizes each log line by attack type. Here we count occurrences across all four categories using the fully parsed and joined dataset.

In [4]:
from pandasql import sqldf

df_labels = df_merged.explode("labels").rename(columns={"labels": "label"})

sqldf(
    "SELECT label, COUNT(*) as total_records FROM df_labels GROUP BY label ORDER BY total_records DESC"
)

,label,total_records
0,attacker_http,35
1,foothold,35
2,dirb,22
3,wpscan,13


## 7. Find Records by Label

Filter the parsed log table to isolate specific attack stages. Each query returns real log fields alongside attack labels.

In [ ]:
# All dirb (directory brute-force) records - shows the IPs and paths being scanned
df_merged[df_merged["labels"].apply(lambda x: "dirb" in x)][
    ["event_timestamp", "client_ip", "log_level", "message", "labels"]
].head(10)

,event_timestamp,log_level,client_ip,message,labels
0,2022-01-24 03:57:26.698653,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.hta_,"[attacker_http, foothold, dirb]"
1,2022-01-24 03:57:26.700701,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htaccess,"[attacker_http, foothold, dirb]"
2,2022-01-24 03:57:26.702901,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htaccess_,"[attacker_http, foothold, dirb]"
3,2022-01-24 03:57:26.704945,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htpasswd,"[attacker_http, foothold, dirb]"
4,2022-01-24 03:57:26.707202,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htpasswd_,"[attacker_http, foothold, dirb]"
5,2022-01-24 03:57:27.362792,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/admin.php' not found or unable to stat,"[attacker_http, foothold, dirb]"
6,2022-01-24 03:57:32.011505,negotiation:error,172.19.131.174,AH00687: Negotiation: discovered file(s) matching request: /var/www/intranet.smith.russellmitchell.com/index (None could be negotiated).,"[attacker_http, foothold, dirb]"
7,2022-01-24 03:57:32.118061,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/info.php' not found or unable to stat,"[attacker_http, foothold, dirb]"
8,2022-01-24 03:57:34.813751,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/phpinfo.php' not found or unable to stat,"[attacker_http, foothold, dirb]"
9,2022-01-24 03:57:36.476812,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/server-status,"[attacker_http, foothold, dirb]"


In [ ]:
# Records tagged with both attacker_http and foothold - high-confidence attacker activity
df_merged[df_merged["labels"].apply(lambda x: "attacker_http" in x and "foothold" in x)][
    ["event_timestamp", "client_ip", "log_level", "message", "labels"]
].head(10)

,event_timestamp,log_level,client_ip,message,labels
0,2022-01-24 03:57:26.698653,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.hta_,"[attacker_http, foothold, dirb]"
1,2022-01-24 03:57:26.700701,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htaccess,"[attacker_http, foothold, dirb]"
2,2022-01-24 03:57:26.702901,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htaccess_,"[attacker_http, foothold, dirb]"
3,2022-01-24 03:57:26.704945,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htpasswd,"[attacker_http, foothold, dirb]"
4,2022-01-24 03:57:26.707202,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htpasswd_,"[attacker_http, foothold, dirb]"
5,2022-01-24 03:57:27.362792,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/admin.php' not found or unable to stat,"[attacker_http, foothold, dirb]"
6,2022-01-24 03:57:32.011505,negotiation:error,172.19.131.174,AH00687: Negotiation: discovered file(s) matching request: /var/www/intranet.smith.russellmitchell.com/index (None could be negotiated).,"[attacker_http, foothold, dirb]"
7,2022-01-24 03:57:32.118061,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/info.php' not found or unable to stat,"[attacker_http, foothold, dirb]"
8,2022-01-24 03:57:34.813751,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/phpinfo.php' not found or unable to stat,"[attacker_http, foothold, dirb]"
9,2022-01-24 03:57:36.476812,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/server-status,"[attacker_http, foothold, dirb]"


In [ ]:
# wpscan records - WordPress vulnerability scanning attempts
df_merged[df_merged["labels"].apply(lambda x: "wpscan" in x)][
    ["event_timestamp", "client_ip", "log_level", "message", "labels"]
].head(10)

,event_timestamp,log_level,client_ip,message,labels
0,2022-01-24 03:57:53.801865,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/searchreplacedb2.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
1,2022-01-24 03:57:54.058025,autoindex:error,172.19.131.174,"AH01276: Cannot serve directory /var/www/intranet.smith.russellmitchell.com/wp-content/uploads/: No matching DirectoryIndex (index.html,index.cgi,index.pl,index.php,index.xhtml,index.htm) found, and server-generated directory index forbidden by Options directive, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
2,2022-01-24 03:57:54.072251,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/emergency.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
3,2022-01-24 03:57:56.986400,php7:error,172.19.131.174,"PHP Fatal error: Uncaught Error: Call to undefined function get_header() in /var/www/intranet.smith.russellmitchell.com/wp-content/themes/go/index.php:15\nStack trace:\n#0 {main}\n thrown in /var/www/intranet.smith.russellmitchell.com/wp-content/themes/go/index.php on line 15, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
4,2022-01-24 03:58:01.182307,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/timthumb.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
5,2022-01-24 03:58:01.182786,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/thumb.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
6,2022-01-24 03:58:08.587535,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/wp-content/themes/thumb.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
7,2022-01-24 03:58:08.590630,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/wp-content/themes/!timtimthumb.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
8,2022-01-24 03:58:08.591131,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/wp-content/themes/!timthumb.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"
9,2022-01-24 03:58:09.170391,php7:error,172.19.131.174,"script '/var/www/intranet.smith.russellmitchell.com/wp-content/timthumb.php' not found or unable to stat, referer: https://intranet.smith.russellmitchell.com","[attacker_http, foothold, wpscan]"


## 8. Records Carrying Only One Label

Lines that triggered exactly one detection rule - useful for validating individual signatures in isolation.

In [8]:
df_single = df_merged[df_merged["labels"].apply(len) == 1].copy()
df_single["label"] = df_single["labels"].apply(lambda x: x[0])

df_single.groupby("label").size().reset_index(name="total").sort_values("total", ascending=False)

,label,total


In [9]:
# Show the actual log entries for single-label records
df_single[["event_timestamp", "client_ip", "log_level", "message", "label"]].head(10)

,event_timestamp,log_level,client_ip,message,label


## 9. What Signatures Does Each Label Fire?

The `rules` field contains a nested dictionary of signature matches. Flatten it to document all sub-fields and how they map to `http.request.sig_match` in our schema.

In [10]:
df_flat = (
    df_merged["rules"]
    .apply(lambda r: list(r.keys()) if isinstance(r, dict) else [])
    .explode()
    .dropna()
)

# Unique signatures fired per label
sig_rows = []
for _, row in df_merged.iterrows():
    if isinstance(row["rules"], dict):
        for label, sigs in row["rules"].items():
            for sig in sigs:
                sig_rows.append({"label": label, "signature": sig})

df_sigs = pd.DataFrame(sig_rows).drop_duplicates()
df_sigs.sort_values("label")

,label,signature
0,attacker_http,attacker.foothold.apache.error
1,attacker_http,attacker.foothold.apache.access_error
2,attacker_http,attacker.foothold.apache.error_substring
3,dirb,attacker.dirb.time
4,foothold,attacker.foothold.apache.error
5,foothold,attacker.foothold.apache.access_error
6,foothold,attacker.foothold.apache.error_substring
7,wpscan,attacker.wpscan.time


## 10. Mapping to the Database Schema

Here's how this data maps to our planned **`http_events`** table in PostgreSQL:

| Raw Field | DB Column | SQL Type | Notes |
|-----------|-----------|----------|-------|
| *(auto-generated)* | `http_event_id` | `SERIAL PRIMARY KEY` | Auto-incrementing ID |
| `line` | `event_id` | `INTEGER NOT NULL` | Original line number from the log file |
| `event_timestamp` | `event_timestamp` | `TIMESTAMP` | Parsed from raw log |
| `log_level` | `log_level` | `VARCHAR(50)` | e.g. `auth_basic:error` |
| `client_ip` | `client_ip` | `INET` | Attacker IP address |
| `message` | `message` | `TEXT` | Full log message |
| `labels` | `http_event_category` | `TEXT[]` | Array of attack-phase labels |
| `rules` | `http_signature_matches` | `JSONB` | Full nested signature match dictionary |
| *(auto-generated)* | `created_at` | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` | Added at insert time |

### The SQL `CREATE TABLE` statement:

```sql
CREATE TABLE http_events (
    http_event_id          SERIAL PRIMARY KEY,
    event_id               INTEGER NOT NULL,
    event_timestamp        TIMESTAMP,
    log_level              VARCHAR(50),
    client_ip              INET,
    message                TEXT,
    http_event_category    TEXT[],
    http_signature_matches JSONB,
    created_at             TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

### 10.0 Type inference assumptions

All SQL type assignments are inferred from observed values in this file. See `type_inference_assumptions.md` for shared team-level rules.

Key decisions for this file:
- `event_id` (line number): INTEGER - line numbers fit well within INT range
- `event_timestamp`: TIMESTAMP - microsecond precision observed in raw timestamps
- `log_level`: VARCHAR(50) - longest observed value is `negotiation:error` (18 chars); 50 gives headroom
- `client_ip`: INET (PostgreSQL) / VARCHAR(45) (MySQL) - supports both IPv4 and IPv6
- `message`: TEXT - variable length, can exceed 255 chars
- `http_event_category`: TEXT[] (PostgreSQL) / JSON (MySQL) - multi-valued label array
- `http_signature_matches`: JSONB (PostgreSQL) / JSON (MySQL) - nested dict, benefits from JSONB indexing in PostgreSQL


### 10.1 Raw DDL


In [ ]:
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE http_events (
    http_event_id          SERIAL PRIMARY KEY,
    event_id               INTEGER NOT NULL,
    event_timestamp        TIMESTAMP,
    log_level              VARCHAR(50),
    client_ip              INET,
    message                TEXT,
    http_event_category    TEXT[],
    http_signature_matches JSONB,
    created_at             TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

mysql_ddl = """
-- MySQL
CREATE TABLE http_events (
    http_event_id          INT AUTO_INCREMENT PRIMARY KEY,
    event_id               INT NOT NULL,
    event_timestamp        DATETIME,
    log_level              VARCHAR(50),
    client_ip              VARCHAR(45),
    message                TEXT,
    http_event_category    JSON,
    http_signature_matches JSON,
    created_at             DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""

print(postgresql_ddl)
print(mysql_ddl)

In [11]:
# Preview of the http_events table as it will look in PostgreSQL
db_preview = df_merged[
    ["line", "event_timestamp", "log_level", "client_ip", "message", "labels", "rules"]
].copy()
db_preview.columns = [
    "event_id",
    "event_timestamp",
    "log_level",
    "client_ip",
    "message",
    "http_event_category",
    "http_signature_matches",
]
db_preview.head(10)

,http_event_id,event_id,event_timestamp,log_level,client_ip,message,http_event_category,http_signature_matches
0,1,2,2022-01-24 03:57:26.698653,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.hta_,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
1,2,3,2022-01-24 03:57:26.700701,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htaccess,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
2,3,4,2022-01-24 03:57:26.702901,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htaccess_,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
3,4,5,2022-01-24 03:57:26.704945,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htpasswd,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
4,5,6,2022-01-24 03:57:26.707202,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.smith.russellmitchell.com/.htpasswd_,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
5,6,7,2022-01-24 03:57:27.362792,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/admin.php' not found or unable to stat,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
6,7,8,2022-01-24 03:57:32.011505,negotiation:error,172.19.131.174,AH00687: Negotiation: discovered file(s) matching request: /var/www/intranet.smith.russellmitchell.com/index (None could be negotiated).,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
7,8,9,2022-01-24 03:57:32.118061,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/info.php' not found or unable to stat,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
8,9,10,2022-01-24 03:57:34.813751,php7:error,172.19.131.174,script '/var/www/intranet.smith.russellmitchell.com/phpinfo.php' not found or unable to stat,"[attacker_http, foothold, dirb]","{'attacker_http': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'foothold': ['attacker.foothold.apache.error', 'attacker.foothold.apache.access_error'], 'dirb': ['attacker.dirb.time']}"
9,10,11,2022-01-24 03:57:36.476812,authz_core:error,172.19.131.174,AH01630: client denied by server configuration: /var/www/intranet.

## 11. Summary Statistics


In [ ]:
from collections import Counter

print("=== Summary ===")
print(f"Total labeled records:   {len(df_merged)}")
print(f"Distinct label categories: {len({lbl for lbls in df_merged['labels'] for lbl in lbls})}")
print(
    f"Time range:              {df_merged['event_timestamp'].min()} to {df_merged['event_timestamp'].max()}"
)
print(f"Distinct client IPs:     {df_merged['client_ip'].nunique()}")
print(f"Columns in DataFrame:    {len(df_merged.columns)}")
print()

print("=== Records with multiple labels ===")
labels_per_row = Counter(len(x) for x in df_merged["labels"])
for n, count in sorted(labels_per_row.items()):
    print(f"  {n} label(s): {count} records")
print()

print("=== log_level distribution ===")
for level, count in df_merged["log_level"].value_counts().items():
    print(f"  {level}: {count}")

## 12. Normalization Observations

Applying the `normalization_rules_sheet.md` checklist to the raw HTTP error log data.

### 12.1 1NF Check

**Multi-valued field:** The `labels` column stores a list of attack-phase tags per record (e.g. `[attacker_http, foothold, dirb]`). This is a **1NF violation** - multiple distinct values packed into a single cell.

**Repeating groups:** The `rules` column is a nested dict mapping each label to a list of matched signatures. Both `labels` and `rules` violate 1NF by storing collections in a single field.

**1NF status: violated.** `labels` (array) and `rules` (nested dict) need to be unpacked. In the `http_events` table these are stored as `TEXT[]` and `JSONB` respectively, which PostgreSQL supports natively but which still represent denormalized collections.

### 12.2 2NF Check

**Primary key:** `http_event_id` is a single-column surrogate PK. Partial dependencies require a composite key, which does not exist here.

**2NF status: satisfied.** Single-column PK makes partial dependencies impossible.

### 12.3 3NF Check

**Transitive dependencies identified:**

| Determinant | Dependent(s) | Notes |
|-------------|-------------|-------|
| `client_ip` | attack context | All 35 records share the same attacker IP (172.19.131.174). In a multi-host dataset, client_ip could determine a host entity. |
| `log_level` | error module | The module prefix (e.g. `authz_core`, `php7`) is embedded in `log_level`. Could be split into `module` and `severity`. |

**3NF status: acceptable for this table.** The dataset is small and single-attacker; splitting `client_ip` into a hosts table or `log_level` into module/severity columns is deferred to the normalization phase.

### 12.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | http_event_id | all attributes | Surrogate PK, trivially determines everything. |
| FD2 | event_id (line) | all attributes | Each line number in this file uniquely identifies one log entry. Candidate key within this file. |
| FD3 | event_timestamp | log_level, client_ip, message | Timestamps are unique per record in this dataset. |


## 13. Key Findings for Schema Design

1. **All 35 records share one attacker IP:** `172.19.131.174` is the sole client across every labeled event. The schema stores this as `INET` for future indexing and multi-host comparisons.

2. **1NF violations in `labels` and `rules`:** Both fields store collections per row. Stored as `TEXT[]` and `JSONB` in PostgreSQL; a fully normalized design would use a junction table for labels and a separate signatures table for rules.

3. **No single-label records:** Every record carries at least 2 labels (`attacker_http` + `foothold` always co-occur). This means label co-occurrence is structural, not incidental, and should be reflected in any normalized label table design.

4. **`log_level` encodes two things:** The field combines the Apache module (e.g. `authz_core`, `php7`, `negotiation`) and severity level (e.g. `error`). A normalized design would split this into `module VARCHAR` and `severity VARCHAR`.

5. **Two attack tool signatures:** `dirb` (directory brute-force, 22 records) and `wpscan` (WordPress scanning, 13 records) are the two sub-tools within the broader foothold phase. The split is clean and non-overlapping in this log.

6. **`message` field is free text:** Contains structured Apache error codes (e.g. `AH01630`) and file paths. Future normalization could extract the error code and path into separate columns for easier querying.
